# EDHCut — Data Overview

Course EDA deliverable (plan task 5.6). Same underlying queries as `edhcut/ingest/qa_report.py`'s markdown report, presented here with `plotly` charts. Reads the already-populated `data/edhcut.db` — no network calls.

In [1]:
import pandas as pd
import plotly.express as px

from edhcut.config import CONFIG
from edhcut.db import connect
from edhcut.ingest import qa_report as qa

pd.set_option("display.max_colwidth", 60)

# Keep the context-manager object itself alive (not just the yielded `conn`) — otherwise the
# generator-based context manager can get garbage-collected mid-notebook, which runs its own
# `finally: conn.close()` and closes the connection out from under later cells.
_db_ctx = connect(CONFIG.paths.db_path)
conn = _db_ctx.__enter__()
slots = qa.resolve_slots(conn)
[s.label for s in slots]


['Krenko, Mob Boss',
 'Kyler, Sigardian Emissary',
 'Yoshimaru, Ever Faithful + Bruse Tarl, Boorish Herder',
 'Yenna, Redtooth Regent',
 'Orysa, Tide Choreographer']

## Per-slot deck counts and card-pool sizes

In [2]:
pool_df = pd.DataFrame([{"slot": s.label, **qa.deck_pool_stats(conn, s)} for s in slots])
display(pool_df)

fig = px.bar(
    pool_df.melt(id_vars="slot", value_vars=["deck_count", "pool_size"], var_name="metric"),
    x="slot", y="value", color="metric", barmode="group",
    title="Decks harvested and distinct-card pool size per commander slot",
)
fig.update_layout(xaxis_title=None)
fig.show()


,slot,deck_count,pool_size
0,"Krenko, Mob Boss",301,1366
1,"Kyler, Sigardian Emissary",300,1675
2,"Yoshimaru, Ever Faithful + Bruse Tarl, Boorish Herder",300,3181
3,"Yenna, Redtooth Regent",300,1909
4,"Orysa, Tide Choreographer",29,654


## Deck size sanity

Every deck should be exactly 99 (single commander) or 98 (partner pair) library cards, reconciled against each deck's own unresolved/banned-card count from the Archidekt harvest audit log (`deck_cards` deliberately excludes cards that fail resolution, e.g. cards later banned in Commander).

In [3]:
archidekt_log_path = CONFIG.paths.logs_dir / "archidekt_harvest_log.txt"
violations = qa.deck_size_violations(conn, archidekt_log_path)

total_decks = conn.execute("SELECT COUNT(*) FROM decks").fetchone()[0]
print(f"{len(violations)} violation(s) out of {total_decks} decks.")

if violations:
    display(pd.DataFrame([{"deck_id": v.deck_id, "expected": v.expected, "actual": v.actual, "url": v.url} for v in violations]))
    print(
        "Both currently-known cases have more than 2 Archidekt \"Commander\"-tagged cards "
        "(a Partner-with/Background chain) — decks.partner_oracle_id only models up to 2, "
        "so this check's 1-or-2-commander assumption doesn't fit them. Their stored "
        "deck_cards counts are internally correct relative to their real commander count."
    )


2 violation(s) out of 1230 decks.


,deck_id,expected,actual,url
0,1072,97,96,https://archidekt.com/decks/8614186/
1,1089,98,96,https://archidekt.com/decks/4456720/


Both currently-known cases have more than 2 Archidekt "Commander"-tagged cards (a Partner-with/Background chain) — decks.partner_oracle_id only models up to 2, so this check's 1-or-2-commander assumption doesn't fit them. Their stored deck_cards counts are internally correct relative to their real commander count.


## Unresolved names per source

Cards each source referenced that don't resolve to our own `cards` table — mostly cards banned/restricted in Commander (excluded by task 5.2's filter) or, for EDHREC, newer prints than the Scryfall snapshot.

In [4]:
raw_name_lookup = qa._oracle_id_to_name_lookup(CONFIG.paths.raw_dir / "oracle_cards.jsonl.gz")
known_oracle_ids = {row[0] for row in conn.execute("SELECT oracle_id FROM cards")}

unresolved_by_source = {
    "Archidekt": qa.unresolved_archidekt_names(archidekt_log_path, set(raw_name_lookup.values())),
    "EDHREC": qa.unresolved_edhrec_names(conn, slots),  # cached session — no new live requests within the 14-day window
    "Tagger bulk": qa.unresolved_tagger_bulk_names(
        CONFIG.paths.raw_dir / "oracle_tags.jsonl.gz", raw_name_lookup, known_oracle_ids
    ),
}

for label, counter in unresolved_by_source.items():
    print(f"\n{label}: {sum(counter.values())} total unresolved occurrences, {len(counter)} distinct names")
    top = pd.DataFrame(counter.most_common(20), columns=["name", "count"])
    display(top)
    if not top.empty:
        px.bar(top, x="name", y="count", title=f"{label} — top unresolved names").show()



Archidekt: 38 total unresolved occurrences, 15 distinct names


,name,count
0,Mana Crypt,8
1,Jeweled Lotus,6
2,Dockside Extortionist,5
3,Liliana the Faultless,3
4,The Queen of Dale,3
5,"Orcrist, Goblin-cleaver",2
6,"Thorin, Mountain-king",2
7,"Sting, Bilbo's Sword",2
8,Bilbo's Gambit,1
9,Thorin Oakenshield,1



EDHREC: 2 total unresolved occurrences, 2 distinct names


,name,count
0,"Orcrist, Goblin-cleaver",1
1,The Queen of Dale,1



Tagger bulk: 24709 total unresolved occurrences, 4036 distinct names


,name,count
0,Mysterious Confluence,36
1,Priest of Possibility,33
2,"Oddric, Lunar Marquis",32
3,Mutable Pupa,31
4,Greater Morphling,30
5,Sproutwatch Dryad,28
6,The Hourglass Coven,26
7,Chandra's Dragonmech,23
8,Seek Bolas's Counsel,23
9,"A-Rowan, Scholar of Sparks // A-Will, Scholar of Frost",23


## EDHREC vs Archidekt top-10 inclusion agreement

Two independently-harvested sources for "what's commonly played with this commander" — how much do their top-10 lists actually overlap?

In [6]:
agreement = qa.edhrec_archidekt_agreement(conn, slots)

overlap_df = pd.DataFrame([{"slot": c.slot_label, "overlap": c.overlap} for c in agreement])
px.bar(overlap_df, x="slot", y="overlap", range_y=[0, 10], title="Top-10 overlap between EDHREC and Archidekt, per slot").show()


def _padded(rows, formatter, n=10):
    formatted = [formatter(*r) for r in rows]
    return formatted + [""] * (n - len(formatted))


for check in agreement:
    print(f"\n{check.slot_label} — overlap {check.overlap}/10")
    side_by_side = pd.DataFrame({
        "EDHREC (inclusion)": _padded(check.edhrec_top10, lambda n, r: f"{n} ({r:.0%})"),
        "Archidekt (deck count)": _padded(check.archidekt_top10, lambda n, c: f"{n} ({c})"),
    })
    display(side_by_side)



Krenko, Mob Boss — overlap 10/10


,EDHREC (inclusion),Archidekt (deck count)
0,Mountain (98%),Goblin Warchief (288)
1,Goblin Warchief (88%),Mountain (286)
2,Skirk Prospector (86%),Skirk Prospector (283)
3,Sol Ring (85%),Impact Tremors (276)
4,Impact Tremors (85%),Goblin Matron (275)
5,Goblin Matron (84%),Sol Ring (271)
6,Goblin Chieftain (77%),Goblin Bombardment (261)
7,Goblin Bombardment (74%),Goblin Chieftain (255)
8,Brightstone Ritual (68%),Brightstone Ritual (240)
9,Pashalik Mons (68%),Pashalik Mons (238)



Kyler, Sigardian Emissary — overlap 8/10


,EDHREC (inclusion),Archidekt (deck count)
0,Plains (99%),Plains (291)
1,Forest (99%),Forest (291)
2,Command Tower (89%),Command Tower (285)
3,Sol Ring (89%),Champion of Lambholt (282)
4,Champion of Lambholt (85%),Heronblade Elite (274)
5,Heronblade Elite (83%),Sol Ring (267)
6,Canopy Vista (79%),Canopy Vista (257)
7,Arcane Signet (77%),"Katilda, Dawnhart Prime (250)"
8,Emeritus of Truce // Swords to Plowshares (77%),Avacyn's Pilgrim (244)
9,Avacyn's Pilgrim (75%),Swords to Plowshares (239)



Yoshimaru, Ever Faithful + Bruse Tarl, Boorish Herder — overlap 2/10


,EDHREC (inclusion),Archidekt (deck count)
0,Command Tower (92%),Plains (284)
1,Emeritus of Truce // Swords to Plowshares (86%),Mountain (275)
2,Flowering of the White Tree (84%),Command Tower (255)
3,Mines of Moria (82%),Sol Ring (226)
4,"Sokenzan, Crucible of Defiance (81%)",Swords to Plowshares (214)
5,"Eiganjo, Seat of the Empire (80%)",Arcane Signet (201)
6,"Merry, Esquire of Rohan (79%)",Clifftop Retreat (172)
7,Minas Tirith (79%),Battlefield Forge (172)
8,Battlefield Forge (78%),Path to Exile (168)
9,"Phelia, Exuberant Shepherd (77%)",Sacred Foundry (165)



Yenna, Redtooth Regent — overlap 8/10


,EDHREC (inclusion),Archidekt (deck count)
0,Forest (93%),Plains (287)
1,Plains (93%),Forest (287)
2,Command Tower (83%),Sanctum Weaver (264)
3,"Sythis, Harvest's Hand (81%)","Sythis, Harvest's Hand (261)"
4,Sanctum Weaver (80%),Command Tower (261)
5,Eidolon of Blossoms (75%),Setessan Champion (247)
6,Utopia Sprawl (75%),Eidolon of Blossoms (237)
7,Setessan Champion (72%),Utopia Sprawl (234)
8,Enchantress's Presence (70%),Canopy Vista (219)
9,Jukai Naturalist (69%),Fertile Ground (217)



Orysa, Tide Choreographer — overlap 6/10


,EDHREC (inclusion),Archidekt (deck count)
0,Island (92%),Island (27)
1,Essence Flux (83%),Essence Flux (26)
2,Sapphire Medallion (78%),Thought Vessel (24)
3,Sol Ring (77%),Sol Ring (24)
4,Mystic Sanctuary (72%),Sapphire Medallion (23)
5,Ghostly Flicker (72%),Ghostly Flicker (22)
6,Reliquary Tower (68%),Mystic Sanctuary (22)
7,Blur (67%),Displacer Kitten (22)
8,Planar Incision (67%),Displace (21)
9,Counterspell (65%),Arcane Signet (21)


## Kyler precon-similarity distribution

Kyler shipped as a precon face card — how much of the harvested corpus is a barely-modified copy of the box list?

In [7]:
kyler_slot = next(s for s in slots if s.names[0].startswith("Kyler"))
hist = qa.precon_similarity_histogram(conn, kyler_slot.commander_key)
hist_df = pd.DataFrame(hist, columns=["bucket", "deck_count"])
display(hist_df)

px.bar(
    hist_df, x="bucket", y="deck_count",
    title="Kyler: deck-to-precon Jaccard similarity distribution",
    labels={"bucket": "Similarity bucket", "deck_count": "# decks"},
).show()


,bucket,deck_count
0,0.0-0.1,20
1,0.1-0.2,145
2,0.2-0.3,82
3,0.3-0.4,29
4,0.4-0.5,15
5,0.5-0.6,4
6,0.6-0.7,3
7,0.7-0.8,1
8,0.8-0.9,0
9,0.9-1.0,1


## Orysa corpus thinness (cold-start test)

Orysa is the roster's deliberately-thin commander (recent release, low EDHREC rank) — the tool needs to degrade gracefully here rather than fabricate confidence.

In [8]:
orysa_slot = next(s for s in slots if s.names[0].startswith("Orysa"))
thin = qa.orysa_thinness_stats(conn, orysa_slot)
print(thin)

comparison_df = pd.DataFrame([
    {
        "slot": s.label,
        "decks_harvested": qa.deck_pool_stats(conn, s)["deck_count"],
        "edhrec_analyzed_decks": qa.edhrec_analyzed_deck_count(conn, s.commander_key),
    }
    for s in slots
])
display(comparison_df)

px.bar(
    comparison_df.melt(id_vars="slot", var_name="metric", value_name="decks"),
    x="slot", y="decks", color="metric", barmode="group", log_y=True,
    title="Corpus size per slot (log scale) — Orysa's thinness in context",
).show()


{'deck_count': 29, 'pool_size': 654, 'edhrec_analyzed_decks': 60}


,slot,decks_harvested,edhrec_analyzed_decks
0,"Krenko, Mob Boss",301,42669
1,"Kyler, Sigardian Emissary",300,5533
2,"Yoshimaru, Ever Faithful + Bruse Tarl, Boorish Herder",300,222
3,"Yenna, Redtooth Regent",300,3854
4,"Orysa, Tide Choreographer",29,60


## Fixture decklist resolution check

The user's own real decklists (`data/fixtures/my_decks/`) — every card must resolve to an `oracle_id`.

In [9]:
fixture_results = qa.check_fixture_decklists(conn, CONFIG.paths.fixtures_dir)
fixture_df = pd.DataFrame([
    {"file": r.file, "total_cards": r.total_cards, "unresolved_count": len(r.unresolved), "unresolved": ", ".join(r.unresolved)}
    for r in fixture_results
])
display(fixture_df)

assert all(len(r.unresolved) == 0 for r in fixture_results), "A fixture decklist has an unresolved card — see table above."
print("All fixture decklists fully resolve.")


,file,total_cards,unresolved_count,unresolved
0,kyler.txt,94,0,
1,orysa.txt,71,0,
2,yenna.txt,74,0,
3,yoshimaru_bruse.txt,85,0,


All fixture decklists fully resolve.


In [10]:
_db_ctx.__exit__(None, None, None)


False